# ML-04 - Search Intelligence Data Contract

Lane: **Content Refresh Scoring**. This notebook writes the contract for my lane's slice of the
warehouse, then proves every claim with a real DuckDB query against the Hugging Face release
(month=2026-03, a mid-panel month - the final month=2026-06 stays sealed as test).


## 1. Unit of analysis + time window

- **One row = one daily performance observation**: a single `(report_date, client_hash_id,
  content_hash_id)` triplet in `fact_content_daily_performance` - one content item's search
  (and, where available, analytics) activity on one calendar day for one client.
- **Table(s) I use:** `fact_content_daily_performance` (primary - partitioned by `month=YYYY-MM`)
  joined to `dim_clients` on `client_hash_id` (for `gsc_data_start` / `ga4_data_start` /
  `ga4_data_available` context only, never as a feature).
- **Time window:** iterate on the mid-panel partition **`month=2026-03`**
  (2026-03-01 → 2026-03-31). The final `month=2026-06` is the sealed test month, and the
  `_sample` table is never used to develop label logic (it *is* the final month).
- **Predict / rank (proxy):** a content-refresh-priority proxy - which content items are
  **losing search visibility** and should be queued for review first. Concretely:
  `is_declining` = GSC impressions in the second half of the month < 80% of impressions in the
  first half, built per content item entirely from March's own daily rows (same idea as
  notebook 03's last30/prev30 split, folded into one month so week 3 stays inside one partition).
- **Deliberately excluded:** GA4 engagement columns (`ga4_sessions`, `engaged_sessions`,
  `scroll_events`) on rows where `ga4_data_available` is not `TRUE`. Those are zero-filled
  placeholders for "not tracked yet," not a real zero - including them un-filtered would teach
  a model that untracked clients have no engagement, which is false.


In [ ]:
# section 1, just writing the contract down as variables so it's easy to grep later
LANE = "content refresh scoring"
UNIT_OF_ANALYSIS = "one row = one (report_date, client_hash_id, content_hash_id) daily observation"
DEV_MONTH = "2026-03"   # iterating here, mid-panel month
TEST_MONTH = "2026-06"  # sealed final month, not touching this for label logic
TARGET_PROXY = "is_declining: 2nd half of month impressions < 80% of 1st half impressions"

print("Lane:", LANE)
print("Unit of analysis:", UNIT_OF_ANALYSIS)
print("Dev month:", DEV_MONTH, "| Sealed test month:", TEST_MONTH)
print("Target proxy:", TARGET_PROXY)

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | **Feature** | Observed search signals, knowable the moment they're logged for a past day |
| `ga4_sessions` / `engaged_sessions` (only where `ga4_data_available IS TRUE`) | **Feature** | Real, already-occurred engagement - but only where the flag confirms it was actually tracked |
| 1st-half-of-month impressions/clicks/position (days 1–15) | **Feature** | Comes entirely from the earlier half of the window, before the 2nd-half outcome exists |
| 2nd-half-of-month impressions | **Label / proxy** | This *is* what `is_declining` is computed from - never a feature |
| `is_declining` | **Label / proxy** | The target itself |
| `report_date`, `client_hash_id`, `content_hash_id` | **Context** | Grouping, joining, per-client splitting only - pseudonyms carry no signal |
| GA4 columns where `ga4_data_available` is `FALSE` | **Excluded** | Zero-filled placeholder, not an observed zero - including them would inject a "not yet tracked" signal disguised as "no engagement" |
| Any product-computed score (`health_score`, `priority_score`, `action_type`) | **Excluded** | Not shipped in this release on purpose, and if it ever were, using it would mean copying FlyRank's existing rule instead of discovering signal |


In [ ]:
# same buckets from the table above, just as a dict so later cells can reference them
FIELD_BUCKETS = {
    "feature": ["gsc_impressions_h1", "gsc_clicks_h1", "gsc_avg_position_h1", "ga4_sessions_h1", "active_days_h1"],
    "label_proxy": ["gsc_impressions_h2", "is_declining"],
    "context": ["report_date", "client_hash_id", "content_hash_id"],
    "excluded": ["ga4_* where ga4_data_available IS FALSE", "health_score/priority_score (not shipped anyway)"],
}

for bucket, fields in FIELD_BUCKETS.items():
    print(bucket, "->", fields)

## 3. Verify with real queries - grain, counts + span, availability - then 5 features + the leakage trap

Connecting DuckDB straight to the Hugging Face release (never downloading the 79M-row table).


In [ ]:
%pip -q install duckdb

import os, getpass
import duckdb

# never paste the token in a cell, repo is public - use colab secret or getpass
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("paste your HF read token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("pointed at:", FACT_MARCH)

In [ ]:
# query 1 - grain check. one row should really be one (report_date, client_hash_id, content_hash_id)
# if this comes back empty, the grain holds

grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("duplicate-grain rows found:", len(grain_probe), "(should be 0)")
grain_probe

In [ ]:
# query 2 - row count + date span for the march slice

counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MARCH}
""").df()

print("march 2026 slice:")
counts

In [ ]:
# query 3 - availability. ga4_data_available IS FALSE means those GA4 columns are just
# zero-filled placeholders (client wasn't tracked yet), not real zero engagement.
# filtering with IS TRUE and checking how many rows actually survive that

availability = con.sql(f"""
    WITH joined AS (
        SELECT f.*, c.ga4_data_start
        FROM {FACT_MARCH} f
        LEFT JOIN {DIM_CLIENTS} c USING (client_hash_id)
    )
    SELECT
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS rows_ga4_not_available,
        COUNT(*) AS total_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_survive
    FROM joined
""").df()

print("rows that survive an IS TRUE availability filter:")
availability

### 5 features (max) - built from March, one per row = one content item

Aggregating each content item's March rows down to a small feature frame, using **only the
first half of the month (days 1–15)** so nothing here overlaps the `is_declining` outcome
window (days 16–end).


In [ ]:
# building the feature frame - per content item, using only the first half of march
# (days 1-15) so nothing overlaps the label window. also computing the label here from
# the second half, but keeping it separate - never going into the actual feature set

feature_frame = con.sql(f"""
    WITH bounds AS (
        SELECT DATE '2026-03-16' AS split_d
    ),
    per_item AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date < b.split_d THEN f.gsc_impressions ELSE 0 END) AS gsc_impressions_h1,
            SUM(CASE WHEN f.report_date < b.split_d THEN f.gsc_clicks ELSE 0 END) AS gsc_clicks_h1,
            AVG(CASE WHEN f.report_date < b.split_d THEN f.gsc_avg_position END) AS gsc_avg_position_h1,
            SUM(CASE WHEN f.report_date < b.split_d AND f.ga4_data_available IS TRUE
                     THEN f.ga4_sessions ELSE 0 END) AS ga4_sessions_h1,
            COUNT(DISTINCT CASE WHEN f.report_date < b.split_d AND f.gsc_impressions > 0
                                 THEN f.report_date END) AS active_days_h1,
            SUM(CASE WHEN f.report_date >= b.split_d THEN f.gsc_impressions ELSE 0 END) AS gsc_impressions_h2
        FROM {FACT_MARCH} f, bounds b
        GROUP BY 1, 2
        HAVING gsc_impressions_h1 >= 10  -- drop near-zero traffic items, too noisy to rank
    )
    SELECT *,
           (gsc_impressions_h2 < 0.8 * gsc_impressions_h1)::INT AS is_declining
    FROM per_item
""").df()

print(len(feature_frame), "content items with a march feature frame")
feature_frame.head()

One line per feature - *knowable at the decision moment because…*

1. **`gsc_impressions_h1`** - knowable, because days 1–15 have already elapsed by the decision
   point (the 16th); it's a completed count, not a forecast.
2. **`gsc_clicks_h1`** - same reasoning: clicks from already-elapsed days.
3. **`gsc_avg_position_h1`** - the average rank Google *already* gave the page over days 1–15;
   nothing here depends on days 16+.
4. **`ga4_sessions_h1`** - engagement already logged in the first half, and only counted where
   `ga4_data_available IS TRUE`, so it's a real observed number, not a placeholder zero.
5. **`active_days_h1`** - how many of the first 15 days actually had impressions; a pattern
   fully contained in the past, before the label window opens.


### 4. The trap - deliberate leakage, on purpose

Adding **one** column derived straight from the label window, training a quick score, watching
it jump toward perfect - then deleting it and keeping the honest number. This is notebook 02's
leakage lesson, replayed here on real warehouse data instead of the starter CSV.


In [ ]:
# honest model first - just the 5 features, nothing from the label window
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ["gsc_impressions_h1", "gsc_clicks_h1", "gsc_avg_position_h1", "ga4_sessions_h1", "active_days_h1"]

model_df = feature_frame.dropna(subset=honest_cols + ["is_declining"])
X, y = model_df[honest_cols], model_df["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print("honest quick score (5 features, first-half only) - AUC:", round(honest_auc, 3))

In [ ]:
# now the trap - adding gsc_impressions_h2 back in as a "feature". this is literally
# the exact number is_declining was computed from, so this is cheating on purpose.
# watch what happens to the score

leaky_cols = honest_cols + ["gsc_impressions_h2"]
leaky_df = feature_frame.dropna(subset=leaky_cols + ["is_declining"])
Xl, yl = leaky_df[leaky_cols], leaky_df["is_declining"]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)

leaky_model = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(Xl_te)[:, 1])
print("leaky score (+ gsc_impressions_h2) - AUC:", round(leaky_auc, 3))
print("jump:", round(honest_auc, 3), "->", round(leaky_auc, 3), "- that's the label leaking in, not real signal")

In [ ]:
# deleting the leaky column, keeping the honest number - this is the one I'd actually defend
assert "gsc_impressions_h2" not in honest_cols
print("leaky column removed.")
print("final honest score for this contract - AUC:", round(honest_auc, 3), "(5 features, all knowable before the label window opens)")

## 4. Data limits

**One named limitation:** this slice can only speak to clients whose March history is already
established - items with `gsc_impressions_h1 < 10` were dropped so the ranking isn't dominated
by noise from near-zero-traffic pages, but that also means the contract is silent on genuinely
low-traffic content, which real refresh queues do still need to triage separately. It also
inherits the warehouse's unbalanced panel: some clients have far less March history than others
(check `dim_clients.gsc_data_start`), so a client with a short history contributes fewer,
noisier first-half observations than one with a full 15 days.


In [ ]:
# section 4 - the limitation, written down as data too, not just prose
LIMITATION = (
    "slice excludes content items with under 10 first-half impressions (noise filter), "
    "so it can't speak to low-traffic refresh decisions, and it inherits the warehouse's "
    "unbalanced per-client history depth"
)
print(LIMITATION)

## Self-check

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.
